# Project Title : Drone Detection System for Surveillance

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Problem Statement : 
### With the increasing use of drones in unauthorized and potentially harmful activities, detecting drones in surveillance environments has become a critical challenge. Traditional monitoring systems struggle to differentiate drones from similar entities such as birds or aircraft, especially under noisy environmental conditions.

### This project aims to develop an intelligent multi-modal detection system that identifies drones using both:

### Visual data (video frames)

### Acoustic signals (audio recordings)

### The system will:

* ### Detect and classify flying objects (drone vs bird vs aircraft)

* ### Use audio signals to validate or reject visual detections

* ### Reduce false positives caused by birds or environmental noise

* ### Generate prioritized alerts for potential threats

In [2]:
import os

audio_path = "/kaggle/input/datasets/partar/drone-detection-dont-upload/Audio/Audio"
video_path = "/kaggle/input/datasets/partar/drone-detection-dont-upload/Video/Video"

audio_files = os.listdir(audio_path)
video_files = os.listdir(video_path)

audio_data = []
video_data = []

# Extract Audio files.

for file in audio_files:
    if file.endswith(".wav"):
        label = file.split('_')[0]
        full_path = os.path.join(audio_path, file)
        audio_data.append((full_path, label))

# Extract Video files.

for file in video_files:
    if file.endswith('.mp4'):
        label = file.split('_')[1]
        full_path = os.path.join(video_path, file)
        video_data.append((full_path, label))

#### **Extract both audio and video file path and save it to "audio_data" and "video_data" respectivly.**

#### **This files are saved in the form of list.**

In [3]:
# Created Datasets.

audio_df = pd.DataFrame(audio_data, columns = ['Path', 'Class'])
video_df = pd.DataFrame(video_data, columns = ['Path', 'Class'])

In [4]:
# make Label "Threat" if class is "DRONE" else "Non-Threat" in both audio and video data

audio_df['Label'] = audio_df['Class'].apply(lambda x : "Threat" if x == 'DRONE' else "Non-Threat")
video_df['Label'] = video_df['Class'].apply(lambda x : "Threat" if x == 'DRONE' else "Non-Threat")

#### **Make label "Threat" if drone detected otherwise "None-Threat"**

In [5]:
audio_df.head()

,Path,Class,Label
0,/kaggle/input/datasets/partar/drone-detection-...,BACKGROUND,Non-Threat
1,/kaggle/input/datasets/partar/drone-detection-...,DRONE,Threat
2,/kaggle/input/datasets/partar/drone-detection-...,DRONE,Threat
3,/kaggle/input/datasets/partar/drone-detection-...,HELICOPTER,Non-Threat
4,/kaggle/input/datasets/partar/drone-detection-...,DRONE,Threat


In [6]:
audio_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Path    90 non-null     object
 1   Class   90 non-null     object
 2   Label   90 non-null     object
dtypes: object(3)
memory usage: 2.2+ KB


#### There are just 90 audio data in audio file.

In [7]:
video_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 283 entries, 0 to 282
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Path    283 non-null    object
 1   Class   283 non-null    object
 2   Label   283 non-null    object
dtypes: object(3)
memory usage: 6.8+ KB


#### There are just 283 video data in video file.

In [8]:
# Audio Processing.

import librosa

def extract_feature(file_path):
    signal, sr = librosa.load(file_path, sr = 22100)  # sr means sample rating, it converts audio into numerical amplitudes.
    
    mfcc = librosa.feature.mfcc(y = signal, sr = sr, n_mfcc = 13)
    mfcc = np.mean(mfcc.T, axis = 0)

    return mfcc

In [9]:
X, y = [], []

for i in range(len(audio_df)):
    file_path = audio_df.iloc[i]['Path']
    label = audio_df.iloc[i]['Label']

    feature = extract_feature(file_path)

    X.append(feature)
    y.append(label)

X = np.array(X)
y = np.array(y)

#### **Above code will converts the audio amplitude into numerical data.**

In [10]:
# Encode label data.

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [11]:
# Split data with 80-20 ratio.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 101)

In [12]:
from sklearn.ensemble import GradientBoostingClassifier

audio_model = GradientBoostingClassifier()
audio_model.fit(X_train, y_train)
y_pred = audio_model.predict(X_test)
print('Confusion Matrix : \n', classification_report(y_test, y_pred))

Confusion Matrix : 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         8

    accuracy                           1.00        18
   macro avg       1.00      1.00      1.00        18
weighted avg       1.00      1.00      1.00        18



#### **Gradient Boost give me realy good accuracy. Maybe bacause the data is realy low.**

In [13]:
import cv2

# Extract Frame from image.

def extract_frame(video_path, max_frames = 50):
    cap = cv2.VideoCapture(video_path)
    frames = []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total_frames // max_frames)

    count = 0
    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        if count % step == 0:
            frame = cv2.resize(frame, (64, 64))
            frame = frame.astype('float32')
            frame /= 255.0
            frames.append(frame)
            

        count += 1

    cap.release()

    if len(frames) < max_frames:
        padding = [np.zeros((64, 64, 3))] * (max_frames - len(frames))
        frames.extend(padding)
        
    return np.array(frames[:max_frames])

In [14]:
X, y = [], []

for i in range(len(video_df)):
    video_path = video_df.iloc[i]['Path']
    label = video_df.iloc[i]['Label']

    frames = extract_frame(video_path)
    X.append(frames)
    y.append(label)

X = np.array(X)
y = np.array(y)

#### **Above code will extract each video from video path, then make video into normalized image frame**

In [15]:
# Encode label data to convert into numerical form.

encode = LabelEncoder()
y = encode.fit_transform(y)

In [16]:
# Split data with 80-20 ratio.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 101)

In [17]:
# Shape of our data

X_train.shape

(226, 50, 64, 64, 3)

#### 226 : no. of data in training dataset (X_train)

#### 50 : no. frame per data

#### 64, 64 : size of frame or image (64 * 64)

#### 3 : colored image (RGB)

In [18]:
# Model training.

from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import TimeDistributed, LSTM, Dense, GlobalAveragePooling2D, SimpleRNN, GRU, Flatten, Bidirectional, Conv2D, BatchNormalization, MaxPooling2D
from tensorflow.keras.optimizers import Adam, AdamW

cnn_model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape = (64, 64, 3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),

    GlobalAveragePooling2D()
])
    
video_model = Sequential([
    TimeDistributed(cnn_model, input_shape = (20, 64, 64, 3)),
    Bidirectional(GRU(128, activation = 'tanh', return_sequences = True)),
    Bidirectional(GRU(64, activation = 'tanh')),
    Dense(128, activation = 'tanh'),
    Dense(2, activation = 'sigmoid')
])

video_model.compile(optimizer = Adam(learning_rate = 5e-4, weight_decay = 1e-3),
                   loss = 'sparse_categorical_crossentropy',
                   metrics = ['accuracy'])

video_model.fit(X_train, y_train, epochs = 5, validation_split = 0.2)
y_pred = video_model.predict(X_test)
y_pred = np.argmax(y_pred, axis = 1)

print('classification report : \n', classification_report(y_test, y_pred))

2026-03-22 18:30:12.048411: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774204212.278372      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774204212.343151      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774204212.882135      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774204212.882185      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774204212.882188      17 computation_placer.cc:177] computation placer alr

Epoch 1/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 124s 11s/step - accuracy: 0.7034 - loss: 0.5703 - val_accuracy: 0.8696 - val_loss: 0.5565
Epoch 2/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 60s 10s/step - accuracy: 0.8879 - loss: 0.2701 - val_accuracy: 0.8478 - val_loss: 0.3728
Epoch 3/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 59s 10s/step - accuracy: 0.9644 - loss: 0.1269 - val_accuracy: 0.9565 - val_loss: 0.1357
Epoch 4/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 61s 10s/step - accuracy: 0.9882 - loss: 0.0502 - val_accuracy: 1.0000 - val_loss: 0.0368
Epoch 5/5
6/6 ━━━━━━━━━━━━━━━━━━━━ 59s 10s/step - accuracy: 0.9949 - loss: 0.0198 - val_accuracy: 1.0000 - val_loss: 0.0103
2/2 ━━━━━━━━━━━━━━━━━━━━ 18s 9s/step
classification report : 
               precision    recall  f1-score   support

           0       1.00      0.95      0.97        37
           1       0.91      1.00      0.95        20

    accuracy                           0.96        57
   macro avg       0.95      0.97      0.96        57
weighted avg       0.97      0.96      0.97 

#### We need two different models to train model on video data. 

#### CNN + RNN : CNN to deal with spatial data and RNN to deal with sequential data. Because video is just sequence of images for machine.

In [19]:
import joblib

video_model.save('video_model.keras')
joblib.dump(audio_model, 'audio_model.pkl')

['audio_model.pkl']

#### Save both models to use it further.